# Lantern Training Data Preparation

*Notebook written by Natascha Barac*

*Based on archived code by Zach Gillis*

*Date last updated: 28 May 2026*

This notebook prepares data for both the *First Cut* filter and *Second Pass* classifier. Details for exactly which data is included in those training sets can be found in their respective training notebooks, `lantern_first_cut_filter.ipynb` and `lantern_second_pass_classifier.ipynb`. In addition, the `README` file of this notebook contains information about the training set currently implemented at ANTARES, as well as links and descriptions of previous training data. 

This notebook is meant to be run on the cloud-based Rubin Science Platform JupyterLab server. (This was last tested on RSP release 29.2.0). 

## Overview

Rubin's difference imaging pipeline subtracts a template image from each visit image, producing a "difference image" where only flux changes appear. Sources detected in these difference images are called DIASources. The full DP1 dataset contains 1.4 million DIASources within our three selected target survey fields. Our goal is to progressively filter this sample down to sources consistent with spatially extended flux differences that could be lensed AGN.

| Field | RA | Dec | Notes |
|---|---|---|---|
| ECDFS | 53.16° | −28.10° | Extended Chandra Deep Field South |
| Galactic | 95.0° | −25.0° | Low galactic latitude field |
| Ecliptic | 37.98° | 7.015° | Low ecliptic latitude field |

To this end, we assume that all of the sources in DP1 data in our target fields are are not lensed AGN, and use this as the non-LAGN portion of our training data. We use LAGN simulations injected into DP1 (ECDFS field only) as our true LAGN sample. This notebook loads and processes these two datasets, and combines them into a single training set to be used for further analysis and model training. Note that it includes only minimal summaries of these two data—-more thorough analyses can be performed using the notebooks located in the `archive` folder, `1_dp1_diasource_characterization.ipynb` and `2_inj_source_characterization.ipynb`. 

## Summary

1. **Load data** — read in all DP1 data in our fields of choice, as well as injected DIASources
2. **Combine data & save** — assign unique IDs to LAGN and non-LAGN. Injected sources are assigned `lens_id`s when they are loaded in; DP1 non-lenses are assigned unique `lens_id`s based on grid-based clustering within a 3 arcsecond radius. These are then combined and saved as `combined_training_data_{version}`
   
---

In [ ]:
version = 'v2.0.7'

# 1. Imports & Data Loading

### Import Libraries

In [ ]:
from lsst.rsp import get_tap_service
from lsst.rsp.service import get_siav2_service
from lsst.rsp.utils import get_pyvo_auth

import lsst.afw.display as afwDisplay
from lsst.afw.image import ExposureF
from lsst.afw.math import Warper, WarperConfig
from lsst.afw.fits import MemFileManager
import lsst.geom as geom

from pyvo.dal.adhoc import DatalinkResults, SodaQuery

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle, Patch
from matplotlib.lines import Line2D
from matplotlib.colors import to_rgba
from astropy import units as u
from astropy.table import Table, vstack
from astropy.coordinates import SkyCoord, search_around_sky
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
import corner
from glob import glob
import os
import yaml
import pandas as pd
from pathlib import Path

import time
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import random

service = get_tap_service("tap")
assert service is not None

sia_service = get_siav2_service("dp1")
assert sia_service is not None

afwDisplay.setDefaultBackend('matplotlib')

import data_processing as dp

### Load DP1 DIASources

`dp.load_dp1` reads data from all three survey fields and merges them into a single Astropy `Table`. Note that for the *Second Pass* training, we use all three survey fields, but for the *First Cut* we only use the ECDFS field. This is because the injected data were only in the ECDFS field, and we do not want the *First Cut* to simply pick up on field differences. We included all fields for the *Second Pass* in order in increase our sample of non-lenses. 

**Options:**
- `fetch_from_server=False` — load from local `.fits` files in `dp1_fields/` (fast, recommended)
- `fetch_from_server=True` — query the TAP service directly (slow, use only to refresh data; must use the first time you run)
- `load_all_fields=True` — load all three survey fields
- `load_all_fields=False` + `field=<field>` — load a single field; `field` options: `'ecdfs'`, `'galactic'`, `'ecliptic'`

### Engineered Features

After loading, `dp.add_engineered_features` appends derived columns used throughout this notebook:

**Notation:** $F_\text{ap}$ = `apFlux`, $F_\text{psf}$ = `psfFlux`, $F_\text{sci}$ = `scienceFlux`, $F_\text{template}$ = `template_flux` (derived); $I_{xx}, I_{yy}, I_{xy}$ = `ixx, iyy, ixy`; $I_{xx}^\text{PSF}, I_{yy}^\text{PSF}, I_{xy}^\text{PSF}$ = `ixxPSF, iyyPSF, ixyPSF`

| Column | Formula | Meaning |
|---|---|---|
| `flux_ext` | $F_\text{ap} / F_\text{psf}$ | Aperture-to-PSF flux ratio |
| `ellip_ext` | $\frac{\sqrt{(I_{xx}-I_{yy})^2+4I_{xy}^2}}{I_{xx}+I_{yy}} - \frac{\sqrt{(I_{xx}^\text{PSF}-I_{yy}^\text{PSF})^2+4(I_{xy}^\text{PSF})^2}}{I_{xx}^\text{PSF}+I_{yy}^\text{PSF}}$ | Difference of source and PSF ellipticities |
| `moment_ext` | $(I_{xx}+I_{yy}) / (I_{xx}^\text{PSF}+I_{yy}^\text{PSF})$ | Source-to-PSF second moment size ratio |
| `template_flux` | $F_\text{sci} - F_\text{psf}$ | Estimated template PSF flux |
| `temp_sci_flux_ratio` | $F_\text{template} / F_\text{sci}$ | Allows removal of moving objects |
| `psf_fwhm` | $(I_{xx}^\text{PSF} I_{yy}^\text{PSF} - I_{xy}^{\text{PSF}^2})^{1/4} \times 2.355 \times 5$ | PSF FWHM in pixels |

In [ ]:
dp1_diasources = dp.load_dp1(fetch_from_server=False, load_all_fields=True, service=service)
dp1_diasources = dp.add_engineered_features(dp1_diasources)

### Load Injected Sources

Loads injection catalogs from Shenming's directory at `/home/sfu/shared_lagn_injection`. Each folder contains:
- `inj_catalog_visit_*.fits` — injection coordinates (`injection_coords`)
- `diaSources_tab_*.csv` — DIASources from visits without injection (`diasources_no_inj`)
- `injected_diaSources_tab_*.csv` — DIASources from visits with injection (`diasources_inj`)

After merging across all folders, `diasources_inj` are cross-matched to `injection_coords`. Matches are assigned `lagn=True`: these are the **positive (label=1)** training samples. We also add a unique `lens_id` so that we can make sure that we keep track of individual lenses, and prevent data leakage.

We loop over each injection version here (2.0, 2.0.1, and 2.0.2) because each of those injection subgroups was injected on the same grid. Thus, we perform the cross-matching on each subset before combining into one large table. These are all version 2.0 lens injections, meaning they were processed in the same way. 

In [ ]:
# create unique lens_id
def make_unique_lens_ids(injection_coords, match_sep_arcsec=3.0):
    coords = SkyCoord(ra=injection_coords['ra'], dec=injection_coords['dec'], unit='deg')
    idx1, idx2, _, _ = search_around_sky(coords, coords, match_sep_arcsec * u.arcsec) #pairs of indices with sep <= match_sep_arcset
    n = len(coords)
    adj = csr_matrix((np.ones(len(idx1)), (idx1, idx2)), shape=(n, n))
    n_unique, labels = connected_components(adj, directed=False) #finds clusters of coords connected by "edge", assigns label
    return labels, n_unique

# Create the 'lagn' and 'lens_id' columns.
# lens_id is the unique lens index; -1 for non-LAGN DIASources.
def lagn_lookup(sources, lookup, inj_unique_ids, max_sep=0.00085):
    src_coords = SkyCoord(ra=sources['ra'], dec=sources['dec'], unit='deg')
    inj_coords = SkyCoord(ra=lookup['ra'], dec=lookup['dec'], unit='deg')
    idx, sep2d, _ = src_coords.match_to_catalog_sky(inj_coords) #finds nearest neighbour for each source
    match   = sep2d.deg < max_sep
    lens_id = np.where(match, inj_unique_ids[idx], -1) #assigns lens_id if matched, otherwise -1
    return match, lens_id

In [ ]:
base_path = Path('/home/sfu/shared_lagn_injection')

inj_versions = ['v2.0', 'v2.0.1', 'v2.0.2']  #v2.0 = lenses 0-1k, v2.0.1 = lenses 1k-2k, v2.0.2 = lenses 2k-3k
site = 'ecdfs'
bands = ['u', 'g', 'r', 'i', 'z', 'y']

# Store processed tables for each version
all_injection_coords_processed = []
all_diasources_no_inj = []
all_diasources_inj_processed = []
all_unmatched_injections = []  # NEW: Store unmatched injections

# Process each injection version SEPARATELY
for inj_version in inj_versions:
    print(f"\n{'='*60}")
    print(f"Processing {inj_version}")
    print(f"{'='*60}\n")
    
    # Collect files for THIS version only
    version_injection_coords = []
    version_diasources_no_inj = []
    version_diasources_inj = []
    
    folders = []
    for band in bands:
        folders.extend(base_path.glob(f'{inj_version}_{site}_{band}_*'))
    
    for folder_path in folders:
        # inj_catalog_visit_*.fits — one file per folder
        for f in sorted(glob(f'{folder_path}/inj_catalog_visit_*.fits')):
            try:
                table = Table.read(f, format='fits')
                table['field'] = site
                version_injection_coords.append(table)
                print(f"  ✓ inj_catalog   {f} ({len(table):,} rows)")
            except Exception as e:
                print(f"  ✗ SKIPPED: {f}\n    {str(e)[:100]}")

        # diaSources_tab_*.csv — one file per folder
        for f in sorted(glob(f'{folder_path}/diaSources_tab_*.csv')):
            try:
                table = Table.read(f, format='csv')
                table['field'] = site
                version_diasources_no_inj.append(table)
                print(f"  ✓ no_inj        {f} ({len(table):,} rows)")
            except Exception as e:
                print(f"  ✗ SKIPPED: {f}\n    {str(e)[:100]}")

        # injected_diaSources_tab_*.csv — one file per folder
        for f in sorted(glob(f'{folder_path}/injected_diaSources_tab_*.csv')):
            try:
                table = Table.read(f, format='csv')
                table['field'] = site
                version_diasources_inj.append(table)
                print(f"  ✓ inj           {f} ({len(table):,} rows)")
            except Exception as e:
                print(f"  ✗ SKIPPED: {f}\n    {str(e)[:100]}")
    
    # Combine tables for THIS version
    print(f"\nCombining tables for {inj_version}...")
    
    injection_coords_version = vstack(version_injection_coords) if version_injection_coords else Table()
    print(f"  injection_coords rows: {len(injection_coords_version):,}")
    
    diasources_no_inj_version = vstack(version_diasources_no_inj) if version_diasources_no_inj else Table()
    print(f"  diasources_no_inj rows: {len(diasources_no_inj_version):,}")
    
    diasources_inj_version = vstack(version_diasources_inj) if version_diasources_inj else Table()
    print(f"  diasources_inj rows: {len(diasources_inj_version):,}")
    
    # Process THIS version: coordinate matching and unique lens IDs
    if len(injection_coords_version) > 0 and len(diasources_inj_version) > 0:
        print(f"\nProcessing coordinate matching for {inj_version}...")
        
        # Create unique lens_ids for THIS version
        inj_unique_ids, n_unique_lenses = make_unique_lens_ids(injection_coords_version)
        print(f"  Total LAGN (after dedup, before matching): {n_unique_lenses}")
        
        # Add a version-specific offset to lens_id to ensure uniqueness across versions
        version_offset = inj_versions.index(inj_version) * 10000  # 0, 10000, 20000
        inj_unique_ids_offset = inj_unique_ids + version_offset
        
        # Match sources to injections for THIS version
        lagn, lens_id = lagn_lookup(diasources_inj_version, injection_coords_version, inj_unique_ids_offset)
        diasources_inj_version['lagn'] = lagn
        diasources_inj_version['lens_id'] = lens_id
        
        matched_lens_ids = np.unique(lens_id[lens_id >= 0])
        print(f"  Total LAGN in DIASources (after matching): {len(matched_lens_ids)}")
        
        # NEW: Find unmatched injections
        # Add lens_id to injection_coords for this version
        injection_coords_version['lens_id'] = inj_unique_ids_offset
        
        # Find which unique lens_ids from injections are NOT in the matched set
        all_injection_lens_ids = np.unique(inj_unique_ids_offset)
        unmatched_mask = ~np.isin(injection_coords_version['lens_id'], matched_lens_ids)
        unmatched_injections = injection_coords_version[unmatched_mask]
        
        n_unmatched_lenses = len(np.unique(unmatched_injections['lens_id']))
        print(f"  Total UNMATCHED LAGN systems: {n_unmatched_lenses}")
        print(f"  Total UNMATCHED injection rows: {len(unmatched_injections)}")
        
        # Store processed results
        all_injection_coords_processed.append(injection_coords_version)
        all_diasources_inj_processed.append(diasources_inj_version)
        all_unmatched_injections.append(unmatched_injections)  # NEW
    
    # Store no_inj sources (these don't need coordinate matching)
    if len(diasources_no_inj_version) > 0:
        all_diasources_no_inj.append(diasources_no_inj_version)

# Now combine all processed versions
print("\n" + "="*60)
print("Combining ALL processed versions...")
print("="*60)

injection_coords = vstack(all_injection_coords_processed) if all_injection_coords_processed else Table()
print(f"\nTotal injection_coords rows: {len(injection_coords):,}")

diasources_no_inj = vstack(all_diasources_no_inj) if all_diasources_no_inj else Table()
print(f"Total diasources_no_inj rows: {len(diasources_no_inj):,}")

diasources_inj = vstack(all_diasources_inj_processed) if all_diasources_inj_processed else Table()
print(f"Total diasources_inj rows: {len(diasources_inj):,}")

# NEW: Combine and save unmatched injections
unmatched_injections = vstack(all_unmatched_injections) if all_unmatched_injections else Table()
print(f"Total unmatched injection rows: {len(unmatched_injections):,}")

if len(unmatched_injections) > 0:
    n_unmatched_lens_systems = len(np.unique(unmatched_injections['lens_id']))
    print(f"Total UNMATCHED lens systems across all versions: {n_unmatched_lens_systems}")
    unmatched_injections.write(f'unmatched_injections_{version}.csv', format='ascii.csv', overwrite=True)
    print(f"  → Saved to unmatched_injections_{version}.csv")

# Add engineered features to combined tables
diasources_no_inj = dp.add_engineered_features(diasources_no_inj)
diasources_no_inj.write(f'diasources_no_inj_{version}.csv', format='ascii.csv', overwrite=True)

diasources_inj = dp.add_engineered_features(diasources_inj)

print(f"\nFinal total MATCHED lens systems across all versions: {len(np.unique(diasources_inj['lens_id'][diasources_inj['lens_id'] >= 0]))}")

#### Check Spatial Distribution / Matching

RA/Dec scatter plot of DIASources from `diasources_inj` and `diasources_no_inj` across all three survey fields, with each field shown in a separate panel. Each panel overlays four categories:

- **DIASources with Source Injection, `lagn=False`** (red) — DIASources from visits where LAGN were injected, but which did not cross-match to an injection coordinate.
- **DIASources with Source Injection, `lagn=True`** (dark red stars) — DIASources from visits where LAGN were injected and which did cross-match to an injection coordinate.
- **DIASources without Source Injection** (blue) — DIASources from visits with *no* source injection.
- **Injection coordinates** (green) — positions where LAGN were injected.

The spatial overlap between injection coordinates (green) and `lagn=True` DIASources (stars) confirms the cross-matching is working correctly.

Ensure that `max_sep` in the `lagn_lookup()` function above is properly calibrated. 

In [ ]:
sites = ['ecdfs', 'ecliptic', 'galactic']
fig, axes = plt.subplots(1, 3, figsize=(18, 6), dpi=300)

for idx, site in enumerate(sites):
    ax = axes[idx]
    
    # Filter data by field
    diasources_inj_site = diasources_inj[diasources_inj['field'] == site]
    diasources_no_inj_site = diasources_no_inj[diasources_no_inj['field'] == site]
    injection_coords_site = injection_coords[injection_coords['field'] == site]
    
    # Split injected sources by lagn status
    diasources_inj_lagn_false = diasources_inj_site[~diasources_inj_site['lagn']]
    diasources_inj_lagn_true = diasources_inj_site[diasources_inj_site['lagn']]
    
    # Plot
    ax.scatter(diasources_inj_lagn_false['ra'], diasources_inj_lagn_false['dec'], 
               c='red', alpha=0.3, label='DIASources, with injection (LAGN = false)', s=20, marker='o')
    ax.scatter(diasources_inj_lagn_true['ra'], diasources_inj_lagn_true['dec'], 
               c='darkred', alpha=0.5, label='DIASources, with injection (LAGN = true)', s=50, marker='*')
    ax.scatter(diasources_no_inj_site['ra'], diasources_no_inj_site['dec'], 
               c='blue', alpha=0.3, label='DIASources, without injection', s=20)
    ax.scatter(injection_coords_site['ra'], injection_coords_site['dec'], 
               c='green', alpha=0.05, label='Injection coordinates', s=20)
    
    ax.set_xlabel('RA (degrees)')
    ax.set_ylabel('Dec (degrees)')
    ax.set_title(f'Field: {site}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Print counts
    print(f"{site}: {len(diasources_inj_lagn_false)} diasources_inj (lagn=False), "
          f"{len(diasources_inj_lagn_true)} diasources_inj (lagn=True), "
          f"{len(diasources_no_inj_site)} diasources_no_inj, "
          f"{len(injection_coords_site)} injection_coords")

plt.tight_layout()
plt.show()

# 2. Combine & Save

We assign unique IDs to non-LAGN (the DP1 data) based on coordinate matching of DIASources. Note that this is not exact, but gives us a good estimate of how many non-LAGN we have in our training data! This is done by splitting up the field into a grid, and clustering in 3as radii within those grids. The grid method is used to speed up the process, but we note that a source may end up being naively split into to regions. 

We then combine these data and save to a single combined training set. 

In [ ]:
#code for coordinate matching non LAGN on grid basis
#currently has transitive matching, which we might want to account for later

from tqdm import tqdm

def assign_ids_grid_matching(df_nonLAGN, grid_size_deg=0.1, radius_arcsec=3):
    """
    Assign unique IDs to non LAGN detections based on spatial clustering.
    Uses connected components to handle transitive matching.
    
    Parameters:
    -----------
    df_nonLAGN : DataFrame with 'ra' and 'dec' columns
    grid_size_deg : size of grid cells in degrees
    radius_arcsec : matching radius in arcseconds
    """
    
    df = df_nonLAGN.copy()
    df['lens_id'] = -1
    df.reset_index(drop=True, inplace=True)
    
    # Create grid cells
    ra_min, ra_max = df['ra'].min(), df['ra'].max()
    dec_min, dec_max = df['dec'].min(), df['dec'].max()
    
    ra_bins = np.arange(ra_min, ra_max + grid_size_deg, grid_size_deg)
    dec_bins = np.arange(dec_min, dec_max + grid_size_deg, grid_size_deg)
    
    # Assign grid cell to each point
    df['ra_bin'] = np.digitize(df['ra'], ra_bins)
    df['dec_bin'] = np.digitize(df['dec'], dec_bins)
    
    next_id = -1
    
    # Process each grid cell independently
    for ra_idx in tqdm(range(len(ra_bins)), desc="Processing RA bins"):
        for dec_idx in range(len(dec_bins)):
            
            # Get points only in THIS cell
            cell_mask = (df['ra_bin'] == ra_idx) & (df['dec_bin'] == dec_idx)
            
            if not cell_mask.any():
                continue
            
            cell_indices = df.index[cell_mask].values
            n_cell = len(cell_indices)
            
            # Convert to SkyCoord
            cell_coords = SkyCoord(
                ra=df.loc[cell_mask, 'ra'].values * u.deg,
                dec=df.loc[cell_mask, 'dec'].values * u.deg
            )
            
            # Find all pairs within radius using search_around_sky
            idx1, idx2, _, _ = search_around_sky(
                cell_coords, 
                cell_coords, 
                radius_arcsec * u.arcsec
            )
            
            # Build adjacency matrix (sparse)
            adj = csr_matrix(
                (np.ones(len(idx1)), (idx1, idx2)), 
                shape=(n_cell, n_cell)
            )
            
            # Find connected components
            n_components, labels = connected_components(adj, directed=False)
            
            # Assign IDs to each component
            for component_id in range(n_components):
                component_mask = (labels == component_id)
                component_indices = cell_indices[component_mask]
                
                # Assign new unique ID
                df.loc[component_indices, 'lens_id'] = next_id
                next_id -= 1
    
    # Clean up temporary columns
    df.drop(columns=['ra_bin', 'dec_bin'], inplace=True)
    
    print(f"\nTotal detections: {len(df)}")
    print(f"Unique objects (lens_id): {np.abs(next_id)}")
    
    return df

def grid_size_suggestion(df, grid_sizes=[0.1, 1.0, 5.0], target_points_per_cell = 1000):
    
    ra_min, ra_max = df['ra'].min(), df['ra'].max()
    dec_min, dec_max = df['dec'].min(), df['dec'].max()
    
    ra_range = ra_max - ra_min
    dec_range = dec_max - dec_min
    
    print(f"RA range: {ra_range:.2f} degrees")
    print(f"Dec range: {dec_range:.2f} degrees")
        
    for grid_size in grid_sizes:
        n_ra_cells = int(np.ceil(ra_range / grid_size))
        n_dec_cells = int(np.ceil(dec_range / grid_size))
        total_cells = n_ra_cells * n_dec_cells
        avg_points_per_cell = len(df) / total_cells
        
        print(f"\nGrid size: {grid_size}°")
        print(f"  Total cells: {total_cells}")
        print(f"  Average points per cell: {avg_points_per_cell:.1f}")
    
    total_points = len(df)
    target_n_cells = total_points / target_points_per_cell
    
    # Assuming roughly square coverage
    cells_per_side = np.sqrt(target_n_cells)
    suggested_grid_size = ra_range / cells_per_side
    
    print(f"\nSuggested grid size: {suggested_grid_size:.2f} degrees")

grid_size_suggestion(dp1_diasources)

In [ ]:
grid_size = 1

# 1. Configuration: Candidate features to include (final feature set can be set in the next notebook)
include_cols = [
    # Other
    'band',
    'centroid_flag',
    'psf_fwhm',
    'snr',
    
    # Flux
    'template_flux',
    'scienceFlux',
    'psfFlux',
    'apFlux',
    'temp_sci_flux_ratio',
    
    # Extended
    'moment_ext',
    'ellip_ext',
    'flux_ext',
    'extendedness',
    'psfChi2',
    'trailFlux',
    'trailLength',
    
    # Dipole
    'isDipole',
    'dipoleFitAttempted',
    'dipoleChi2',
    'dipoleFluxDiffErr',
    'dipoleMeanFlux',
    'dipoleMeanFluxErr',
    'dipoleLength',
    
    # Centroid
    'x_y_err',
]

# 2. Preprocessing Functions
def normalize_bands_and_flags(df):
    """Normalizes band strings and converts flag columns. Returns a copy."""
    df = df.copy()
    if 'band' in df.columns:
        # Handle both FilterLabel(band="g", ...) and plain 'g' formats
        extracted = df['band'].astype(str).str.extract(r"band=['\"]([ugrizy])['\"]", expand=False)
        # Fall back to bare letter for rows already in plain format
        plain = df['band'].astype(str).str.extract(r'^([ugrizy])$', expand=False)
        df['band'] = extracted.fillna(plain)
        # XGBoost requires categorical columns to be the 'category' dtype
        df['band'] = df['band'].astype('category')
    flag_cols = [c for c in df.columns if 'flag' in c.lower() or c in ['isDipole']]
    for col in flag_cols:
        df[col] = df[col].astype(bool)
    return df

def filter_columns(df, features):
    """Reduces to only the requested feature columns."""
    existing_cols = [c for c in features if c in df.columns]
    return df[existing_cols]

print("Preparing datasets...")

# 3. Prepare Training Data
false_df_unfiltered = normalize_bands_and_flags(dp1_diasources.to_pandas())
true_df_unfiltered  = normalize_bands_and_flags(diasources_inj[diasources_inj['lagn']].to_pandas())

# Assign IDs to non LAGN (false) data
false_df_with_ids = assign_ids_grid_matching(false_df_unfiltered, grid_size_deg=grid_size, radius_arcsec=3)

# Filter columns for both
# false_df = filter_columns(false_df_with_ids, include_cols)
# true_df  = filter_columns(true_df_unfiltered, include_cols)

#OR don't filter columns! currently doing this
false_df = false_df_with_ids.copy()
true_df = true_df_unfiltered.copy()

# Combine the dataframes
combined_data = pd.concat([
    false_df.assign(
        label=0,
        lens_id=false_df_with_ids['lens_id'].values,
        ra=false_df_unfiltered['ra'].values,
        dec=false_df_unfiltered['dec'].values
    ),
    true_df.assign(
        label=1, 
        lens_id=true_df_unfiltered['lens_id'].values,
        ra=true_df_unfiltered['ra'].values,
        dec=true_df_unfiltered['dec'].values
    ),
], ignore_index=True)

# combined_data.to_csv(f'combined_training_data_{version}.csv', index=False)
combined_data.to_csv(f'combined_training_data_{version}_all_cols.csv', index=False)    #currently ALL COLUMNS
print(f"✓ Combined data saved (Version: {version}  |  Rows: {len(combined_data)})")